In [56]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as sm
import sklearn.preprocessing as preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

In [57]:
df = pd.read_csv('/content/dataset_final.csv')


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14268 entries, 0 to 14267
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   award_id                          14268 non-null  object 
 1   release_id                        14268 non-null  object 
 2   fecha_adj                         14268 non-null  object 
 3   monto_adj                         14268 non-null  int64  
 4   log_monto                         14268 non-null  float64
 5   anio                              14268 non-null  int64  
 6   mes                               14268 non-null  int64  
 7   contract_id                       14264 non-null  object 
 8   fecha_firma                       14264 non-null  object 
 9   plazo_dias                        14264 non-null  float64
 10  ruc                               14268 non-null  object 
 11  proveedor                         14268 non-null  object 
 12  prov

In [59]:
datos = df.copy()

# **Variable objetivo**
La variable obejtivo es "target_ampliacion_monto" basandose en el objetivo del trabajo.

Se utilizaran modelos de categorizacion para poder identificar contratos que podrían requerir un seguimiento más cercano durante su ejecución apartir de su adjudicacion.

Se eliminan algunas variables que no son relevantes para el modelo o que no son soportados con estandarizacion o algunos algoritmos.

In [60]:
columnas_drop = [
    'award_id',
    'release_id',
    'contract_id',
    'ruc',
    'proveedor',
    'fecha_fin_lic',
    'fecha_ini_lic',
    'fecha_firma',
    'fecha_pub_lic',
    'n_enmiendas',
    'hubo_ampl_plazo',
    'monto_ampliado_total',
    'dias_ampliacion_total',
    'dias_hasta_enmienda',
    'ratio_ampliacion'
]

datos = datos.drop(
    columns=columnas_drop,
    errors='ignore'
)

In [61]:

datos['fecha_adj'] = pd.to_datetime(
    datos['fecha_adj'],
    errors='coerce'
)

datos = datos.sort_values('fecha_adj').reset_index(drop=True)

In [62]:
objetivo = 'target_ampliacion_monto'

# **Revision de datos**
tratamiento de valores faltantes o irregulares

In [63]:
for col in datos.columns:
  print(f"{col}")
  print("Cantidad de nulos:", datos[col].isna().sum())
  print("Cantidad de ceros:" + str((datos[col] == 0).sum()) + "\n")

fecha_adj
Cantidad de nulos: 0
Cantidad de ceros:0

monto_adj
Cantidad de nulos: 0
Cantidad de ceros:1

log_monto
Cantidad de nulos: 0
Cantidad de ceros:1

anio
Cantidad de nulos: 0
Cantidad de ceros:0

mes
Cantidad de nulos: 0
Cantidad de ceros:0

plazo_dias
Cantidad de nulos: 4
Cantidad de ceros:40

prov_n_previas
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_monto_previo_acum
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_antiguedad_dias
Cantidad de nulos: 0
Cantidad de ceros:3715

n_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

monto_estimado_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

det_metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

categoria_principal
Cantidad de nulos: 0
Cantidad de ceros:0

monto_licitacion
Cantidad de nulos: 0
Cantidad de ceros:3

monto_presupuesto
Cantidad de nulos: 813
Cantidad de ceros:0

ratio_adj_vs_licitacion
Cantidad de nulos: 0
Cantidad de ceros:1

brecha_li

In [64]:
median = [
    'plazo_dias',
    'dur_licitacion_dias',
    'dur_consultas_dias',
    'dias_lic_a_adj',
    'monto_presupuesto'
]

In [65]:
datos['brecha_licitacion_vs_presupuesto'] = datos['monto_presupuesto']/datos['monto_licitacion']
datos[datos['dur_licitacion_dias'] < 0 ]['dur_licitacion_dias'] = datos['dur_licitacion_dias'].median()
datos[datos['dur_consultas_dias'] < 0 ]['dur_consultas_dias'] = datos['dur_consultas_dias'].median()
datos[datos['velocidad_proceso_dias'] < 0 ]['velocidad_proceso_dias'] = datos['velocidad_proceso_dias'].median()
datos['brecha_licitacion_vs_presupuesto'].replace([np.inf, -np.inf], np.nan, inplace=True)
datos['brecha_licitacion_vs_presupuesto'].fillna(0, inplace=True)
datos['ratio_adj_vs_licitacion'].replace([np.inf, -np.inf], np.nan, inplace=True)
datos['ratio_adj_vs_licitacion'].fillna(0, inplace=True)

/tmp/ipykernel_959/3395749771.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datos[datos['dur_licitacion_dias'] < 0 ]['dur_licitacion_dias'] = datos['dur_licitacion_dias'].median()
/tmp/ipykernel_959/3395749771.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datos[datos['dur_consultas_dias'] < 0 ]['dur_consultas_dias'] = datos['dur_consultas_dias'].median()
/tmp/ipykernel_959/3395749771.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [66]:
for col in datos.columns:
  print(f"{col}")
  print("Cantidad de nulos:", datos[col].isna().sum())
  print("Cantidad de ceros:" + str((datos[col] == 0).sum()) + "\n")

fecha_adj
Cantidad de nulos: 0
Cantidad de ceros:0

monto_adj
Cantidad de nulos: 0
Cantidad de ceros:1

log_monto
Cantidad de nulos: 0
Cantidad de ceros:1

anio
Cantidad de nulos: 0
Cantidad de ceros:0

mes
Cantidad de nulos: 0
Cantidad de ceros:0

plazo_dias
Cantidad de nulos: 4
Cantidad de ceros:40

prov_n_previas
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_monto_previo_acum
Cantidad de nulos: 0
Cantidad de ceros:3594

prov_antiguedad_dias
Cantidad de nulos: 0
Cantidad de ceros:3715

n_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

monto_estimado_lotes
Cantidad de nulos: 0
Cantidad de ceros:813

metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

det_metodo_contratacion
Cantidad de nulos: 0
Cantidad de ceros:0

categoria_principal
Cantidad de nulos: 0
Cantidad de ceros:0

monto_licitacion
Cantidad de nulos: 0
Cantidad de ceros:3

monto_presupuesto
Cantidad de nulos: 813
Cantidad de ceros:0

ratio_adj_vs_licitacion
Cantidad de nulos: 0
Cantidad de ceros:4

brecha_li

In [67]:
columnas_no_predictoras = [
    objetivo,
    'fecha_adj'
]

X = datos.drop(columns=columnas_no_predictoras, errors='ignore')

y = datos[objetivo]

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print("\nVariables predictoras:")
print(X.columns.tolist())
X.shape


Dimensiones de X: (14268, 24)
Dimensiones de y: (14268,)

Variables predictoras:
['monto_adj', 'log_monto', 'anio', 'mes', 'plazo_dias', 'prov_n_previas', 'prov_monto_previo_acum', 'prov_antiguedad_dias', 'n_lotes', 'monto_estimado_lotes', 'metodo_contratacion', 'det_metodo_contratacion', 'categoria_principal', 'monto_licitacion', 'monto_presupuesto', 'ratio_adj_vs_licitacion', 'brecha_licitacion_vs_presupuesto', 'criterio_adj', 'det_criterio_adj', 'dur_licitacion_dias', 'dur_consultas_dias', 'velocidad_proceso_dias', 'dias_pub_a_adj', 'dias_lic_a_adj']


(14268, 24)

In [68]:
categoricas = [
    'metodo_contratacion', 'det_metodo_contratacion', 'categoria_principal', 'criterio_adj', 'det_criterio_adj'
]
X[categoricas].describe()
for col in categoricas:
    print(f"\n--- {col} ---")
    print(X[col].value_counts(dropna=False))


--- metodo_contratacion ---
metodo_contratacion
open      14001
direct      267
Name: count, dtype: int64

--- det_metodo_contratacion ---
det_metodo_contratacion
Contratacion Directa                                            6651
Licitacion Publica Nacional                                     2913
Concurso de Ofertas                                             2764
Locacion de Inmuebles                                            846
Contratacion por Excepcion                                       813
Licitacion Publica Internacional                                  56
BID - Contratacion Directa (CD/SD)                                55
BID - Contratacion por Excepcion                                  29
BID - Concurso de Ofertas (CO/CP)                                 28
BID - Licitacion Publica Nacional                                 16
Seleccion sobre la base la comparacion de las calificaciones      15
BM - Concurso de Ofertas - CO                                     13
BID - Li

In [69]:
X_encoded_lineal_logistic = pd.get_dummies(X, columns=categoricas, drop_first=True)
print("Dimensiones de desarrollo codificado:", X_encoded_lineal_logistic.shape)
X_encoded_lineal_logistic.info()

Dimensiones de desarrollo codificado: (14268, 51)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14268 entries, 0 to 14267
Data columns (total 51 columns):
 #   Column                                                                                Non-Null Count  Dtype  
---  ------                                                                                --------------  -----  
 0   monto_adj                                                                             14268 non-null  int64  
 1   log_monto                                                                             14268 non-null  float64
 2   anio                                                                                  14268 non-null  int64  
 3   mes                                                                                   14268 non-null  int64  
 4   plazo_dias                                                                            14264 non-null  float64
 5   prov_n_previas                 

# **Division**
Se divide las filas cronologicamente de acuerdo a fecha_adj, 80% para el desarrollo y 20% para la prueba

In [70]:
fechas = datos['fecha_adj']
corte = int(len(datos) * 0.80)

X_desarrollo = X[:corte].copy()
y_desarrollo = y.iloc[:corte].copy()

X_prueba = X[corte:].copy()
y_prueba = y.iloc[corte:].copy()

# 9. Mostrar los resultados
print("\nDimensiones:")
print("X desarrollo:", X_desarrollo.shape)
print("y desarrollo:", y_desarrollo.shape)
print("X prueba:", X_prueba.shape)
print("y prueba:", y_prueba.shape)

print("\nPeríodo de desarrollo:")
print(fechas.iloc[0], "a", fechas.iloc[corte - 1])

print("\nPeríodo de prueba:")
print(fechas.iloc[corte], "a", fechas.iloc[-1])

print("\nDistribución del objetivo en desarrollo:")
print(y_desarrollo.value_counts(normalize=True))

print("\nDistribución del objetivo en prueba:")
print(y_prueba.value_counts(normalize=True))


Dimensiones:
X desarrollo: (11414, 24)
y desarrollo: (11414,)
X prueba: (2854, 24)
y prueba: (2854,)

Período de desarrollo:
2010-06-14 21:03:59 a 2011-12-06 18:27:04

Período de prueba:
2011-12-06 18:35:06 a 2013-03-05 16:00:00

Distribución del objetivo en desarrollo:
target_ampliacion_monto
0.0    0.886806
1.0    0.113194
Name: proportion, dtype: float64

Distribución del objetivo en prueba:
target_ampliacion_monto
0.0    0.915557
1.0    0.084443
Name: proportion, dtype: float64


Aplicar One-Hot Encoding a las variables categóricas, si bien det_metodo_contratacion tiene muchas variables unicas, todavia se considera aceptable la aplicacion del metodo de encoding al tener una gran cantidad de filas para el entrenamiento.
El metodo de one hot enconding se usara solo para los modelos lineales y logisticos pero para modelos de arboles se usara category al ser mas eficiente al no obligar al modelo a fragmentar el arbol de decisiones tratando cada valor unico del one-hot encoding, en cambio, un modelo como xgboost agrupara todo los valores en una sola columna y gracias a su funcion hist que funciona con histograma podra evaluar particiones optimas, mejorando las velocidades de inferencia y haciendo mas estable el entrenamiento

La Estandarizacion de manera similar solo se aplicara a los modelos lineales y logisticos, no asi para los modelos de arboles, porque no los necesitan

In [71]:
# Calcular medianas luego de la division para que no haya leakage
medianas_desarrollo = X_desarrollo[median].median()

# Imputar usando solo esa información del pasado
for col in median:
    X_desarrollo[col] = X_desarrollo[col].fillna(medianas_desarrollo[col])
    X_prueba[col] = X_prueba[col].fillna(medianas_desarrollo[col])

In [77]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import xgboost as xgb
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score

# Definir columnas categóricas y numéricas
columnas_cat = ['metodo_contratacion', 'det_metodo_contratacion', 'categoria_principal', 'criterio_adj', 'det_criterio_adj']
columnas_num = [col for col in X_desarrollo.columns if col not in columnas_cat]

# Convertir variables object a categorias para mejorar el entrenamiento
X_desarrollo_xgb = X_desarrollo.copy()
X_prueba_xgb = X_prueba.copy()
for col in columnas_cat:
    X_desarrollo_xgb[col] = X_desarrollo_xgb[col].astype('category')
    X_prueba_xgb[col] = X_prueba_xgb[col].astype('category')

# Balanceo de clases con SMOTE-NC debido a que tenemos menos de 10% de variables objetivos
smote_nc = SMOTENC(categorical_features=columnas_cat, random_state=42)
X_train_res, y_train_res = smote_nc.fit_resample(X_desarrollo_xgb, y_desarrollo)

# Parametros conservadores para XGBoost
param_grid_xgb = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100]
}

# Modelo base de xgboost
modelo_xgb_base = xgb.XGBClassifier(
    tree_method='hist',
    enable_categorical=True,
    random_state=42
)

# Ajuste de hiperparametros de xgboost con GridSearchCV
grid_xgb = GridSearchCV(
    estimator=modelo_xgb_base,
    param_grid=param_grid_xgb,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train_res, y_train_res)
modelo_xgb = grid_xgb.best_estimator_
print("Mejores parámetros para XGBoost:", grid_xgb.best_params_)
print("Mejor score F1 en validación cruzada para XGBoost:", grid_xgb.best_score_)
print("1) Modelo XGBoost optimizado y entrenado exitosamente.")

# Preprocesamiento con OHE y estandarización para modelos lineales
preprocessor_lineal = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), columnas_cat),
        ('num', StandardScaler(), columnas_num)
    ]
)

# --- 2. Pipeline y Ajuste para Regresión Lineal Múltiple ---
pipeline_lineal = Pipeline([
    ('preprocesador', preprocessor_lineal),
    ('modelo', LinearRegression())
])
pipeline_lineal.fit(X_train_res, y_train_res)
print("2) Pipeline de Regresión Lineal entrenado exitosamente.")

# --- 3. Pipeline y Optimización para Regresión Logística ---
pipeline_logistica_base = Pipeline([
    ('preprocesador', preprocessor_lineal),
    ('modelo', LogisticRegression(random_state=42, max_iter=1000))
])

param_grid_logistica = {
    'modelo__C': [0.01, 0.1, 1.0, 10.0]
}

grid_logistica = GridSearchCV(
    estimator=pipeline_logistica_base,
    param_grid=param_grid_logistica,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_logistica.fit(X_train_res, y_train_res)
pipeline_logistica = grid_logistica.best_estimator_
print("Mejores parámetros para Regresión Logística:", grid_logistica.best_params_)
print("Mejor score F1 en validación cruzada para Regresión Logística:", grid_logistica.best_score_)
print("3) Pipeline de Regresión Logística optimizado y entrenado exitosamente con datos balanceados.")

# --- 4. Modelo Base por Media de Categoría Principal ---
# Calculamos la tasa promedio de ampliación (media del target) por cada categoría principal usando solo el set de desarrollo
df_desarrollo_temp = X_desarrollo.copy()
df_desarrollo_temp['target'] = y_desarrollo
medias_categoria = df_desarrollo_temp.groupby('categoria_principal')['target'].mean().to_dict()
media_global_desarrollo = y_desarrollo.mean()

# Función para predecir usando el promedio del grupo
def predecir_modelo_base(df_test):
    return df_test['categoria_principal'].map(medias_categoria).fillna(media_global_desarrollo).values

predicciones_base = predecir_modelo_base(X_prueba)
predicciones_base_clase = (predicciones_base >= 0.5).astype(int)

# Evaluaciones utilizando el conjunto de prueba con el formato de datos adecuado
accuracy_xgb = accuracy_score(y_prueba, modelo_xgb.predict(X_prueba_xgb))

# Regresión lineal devuelve valores continuos, aplicamos un umbral de 0.5 para poder medir accuracy
predicciones_lineales_continuas = pipeline_lineal.predict(X_prueba)
predicciones_lineales_clase = (predicciones_lineales_continuas >= 0.5).astype(int)

accuracy_lineal = accuracy_score(y_prueba, predicciones_lineales_clase)
accuracy_logistica = accuracy_score(y_prueba, pipeline_logistica.predict(X_prueba))
accuracy_base = accuracy_score(y_prueba, predicciones_base_clase)

print("\n--- MÉTRICAS DE EVALUACIÓN EN EL CONJUNTO DE PRUEBA ---")
print("El accuracy de regresion base es: {:.2f}".format(accuracy_base))
print("El accuracy de xgboost es: {:.2f}".format(accuracy_xgb))
print("El accuracy de regresion lineal multiple es: {:.2f}".format(accuracy_lineal))
print("El accuracy de regresion logistica es: {:.2f}".format(accuracy_logistica))

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Mejores parámetros para XGBoost: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 100}
Mejor score F1 en validación cruzada para XGBoost: 0.6993043827631474
1) Modelo XGBoost optimizado y entrenado exitosamente.
2) Pipeline de Regresión Lineal entrenado exitosamente.
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Mejores parámetros para Regresión Logística: {'modelo__C': 10.0}
Mejor score F1 en validación cruzada para Regresión Logística: 0.7068018675462278
3) Pipeline de Regresión Logística optimizado y entrenado exitosamente con datos balanceados.

--- MÉTRICAS DE EVALUACIÓN EN EL CONJUNTO DE PRUEBA ---
El accuracy de regresion base es: 0.92
El accuracy de xgboost es: 0.90
El accuracy de regresion lineal multiple es: 0.82
El accuracy de regresion logistica es: 0.83


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcular la correlación de todas las variables numéricas con la variable objetivo
print("--- Correlación de las variables con el Target (target_ampliacion_monto) ---")
# Creamos un dataset temporal combinando X_desarrollo y y_desarrollo
temp_df = X_desarrollo.copy()
temp_df['target'] = y_desarrollo

# Seleccionamos numéricas
numeric_cols = temp_df.select_dtypes(include=[np.number]).columns
correlations = temp_df[numeric_cols].corr()['target'].sort_values(ascending=False)
print(correlations)

# 2. Extraer y graficar la importancia de características (Feature Importance) de XGBoost
if 'modelo_xgb' in globals():
    importances = modelo_xgb.feature_importances_
    features = X_train_res.columns
    df_importance = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(12, 6))
    sns.barplot(x='Importance', y='Feature', data=df_importance.head(15), palette='viridis')
    plt.title('Top 15 Características más importantes en XGBoost (Posible Leakage)')
    plt.xlabel('Importancia')
    plt.ylabel('Variable')
    plt.tight_layout()
    plt.show()

    print("\n--- Top 10 variables más importantes en el modelo XGBoost ---")
    print(df_importance.head(10))
else:
    print("El modelo XGBoost no está disponible en el espacio de nombres.")